# CrisisLexT6

Domain Adaptation with Adversarial Training and Graph Embeddings

---

## A. Original description


This dataset contains tweets labeled by crowdsourcing workers. Each tweet is accompanied by a label
, which is the result of the majority voting among at least 3 crowdsourcing workers.

There is one file per crisis, for each of the following disasters:

* [Sandy Hurricane 2012](https://en.wikipedia.org/wiki/Hurricane_Sandy)
* [Oklahoma Tornado Season 2013](https://en.wikipedia.org/wiki/2013_Moore_tornado)
* [West Texas Explosion 2013](https://en.wikipedia.org/wiki/West_Fertilizer_Company_explosion)
* [Alberta Floods 2013](https://en.wikipedia.org/wiki/2013_Alberta_floods)
* [Boston Bombings 2013](https://en.wikipedia.org/wiki/Boston_Marathon_bombings)
* [Queensland Floods 2013](https://en.wikipedia.org/wiki/January_2013_Eastern_Australia_floods)

On-topic/Off-topic files: `[disasters]-ontopic_offtopic.csv`

**Contents:**
Each file contains approximately 10,000 tweets. 50% of these tweets were
sampled from the geo-based sample, and 50% from the keywords-based sample.
These two samples are described in [Olteanu et al. 2014].

**Labels:**
These files contain labels indicating if a tweet is on-topic (related to
the crisis at hand), or off-topic (not related to it).

**File format:**
One tweet per line with the following comma-separated fields:
tweet id, tweet text, tweet label

### Questions/inquiries

[Olteanu et al. 2014] Alexandra Olteanu, Carlos Castillo, Fernando Diaz, Sarah Vieweg: "CrisisLex: A Lexicon for Collecting and Filtering Microblogged Communications in Crises". ICWSM 2014.

For inquiries please contact [Alexandra Olteanu](mailto:alexandra.olteanu@epfl.ch), or Carlos Castillo, or Fernando Diaz, or Sarah Vieweg.

### Version history

 * 2014-10-26: v1.0, initial release containing labeled tweets only.

## B. Overview

In [1]:
from pathlib import Path
import os
from dotenv import load_dotenv
load_dotenv()
datasets_path = Path(os.getenv("DATASETS_PATH")) / "Datasets"

Loading the `2012_Sandy_Hurricane-ontopic_offtopic.csv` file

In [20]:
import pandas as pd

dataset_path = datasets_path / 'CrisisLexT6' / 'CrisisLexT6'

filepath = dataset_path / '2012_Sandy_Hurricane' / '2012_Sandy_Hurricane-ontopic_offtopic.csv'

if not filepath.exists():
    raise FileNotFoundError(f'Not found: {filepath}')

df = pd.read_csv(filepath)
df['tweet_id'] = df['tweet id'].str.strip("'").astype('int64')
df.drop(columns=['tweet id'], inplace=True)

In [21]:
df.head()

,tweet,label,tweet_id
0,I've got enough candles to supply a Mexican fa...,off-topic,262596552399396864
1,Sandy be soooo mad that she be shattering our ...,on-topic,263044104500420609
2,@ibexgirl thankfully Hurricane Waugh played it...,off-topic,263309629973491712
3,@taos you never got that magnificent case of B...,off-topic,263422851133079552
4,"I'm at Mad River Bar &amp; Grille (New York, N...",off-topic,262404311223504896


In [22]:
csv_array = df.values

print(f'Loaded {len(df)} rows from: {filepath}')
print(csv_array[:5])

Loaded 10008 rows from: /Users/nhut/Library/CloudStorage/GoogleDrive-dhnhut@gmail.com/My Drive/0. 📌 UoA/0. Research Project/Datasets/CrisisLexT6/CrisisLexT6/2012_Sandy_Hurricane/2012_Sandy_Hurricane-ontopic_offtopic.csv
[["I've got enough candles to supply a Mexican family" 'off-topic'
  262596552399396864]
 ['Sandy be soooo mad that she be shattering our doors and shiet #HurricaneSandy'
  'on-topic' 263044104500420609]
 ['@ibexgirl thankfully Hurricane Waugh played it cool and waited this one out. Ready to go at any moment tho.'
  'off-topic' 263309629973491712]
 ['@taos you never got that magnificent case of Burgundy I sent you to thank you for your tweets?'
  'off-topic' 263422851133079552]
 ["I'm at Mad River Bar &amp; Grille (New York, NY) http://t.co/VSiZrzKP"
  'off-topic' 262404311223504896]]


## C. Preprocess

Combine all train, dev, test files from both events to one

In [25]:
import json

files = [
    {
        'filename': '2012_Sandy_Hurricane/2012_Sandy_Hurricane-ontopic_offtopic.csv',
        'country': 'US',
        'event_type': 'hurricane',
        'year': 2012,
    },
    {
        'filename': '2013_Alberta_Floods/2013_Alberta_Floods-ontopic_offtopic.csv',
        'country': 'Canada',
        'event_type': 'flood',
        'year': 2013,
    },
    {
        'filename': '2013_Boston_Bombings/2013_Boston_Bombings-ontopic_offtopic.csv',
        'country': 'US',
        'event_type': 'bombing',
        'year': 2013,
    },
    {
        'filename': '2013_Oklahoma_Tornado/2013_Oklahoma_Tornado-ontopic_offtopic.csv',
        'country': 'US',
        'event_type': 'tornado',
        'year': 2013,
    },
    {
        'filename': '2013_Queensland_Floods/2013_Queensland_Floods-ontopic_offtopic.csv',
        'country': 'Australia',
        'event_type': 'flood',
        'year': 2013,
    },
    {
        'filename': '2013_West_Texas_Explosion/2013_West_Texas_Explosion-ontopic_offtopic.csv',
        'country': 'US',
        'event_type': 'explosion',
        'year': 2013,
    }
]

df_list = []

for file in files:
    filename = file['filename']
    filepath = dataset_path / filename
    if not filepath.exists():
        print(f'Not found: {filepath}')
        continue

    df = pd.read_csv(filepath)
    df['tweet_id'] = df['tweet id'].str.strip("'").astype('int64')
    df.drop(columns=['tweet id'], inplace=True)
    df['event_type'] = file['event_type']
    df['country'] = file['country']
    df['year'] = file['year']
    # Store source filename as JSON text in `meta`.
    df['meta'] = json.dumps({'file_name': filename})
    print(f'Loaded {len(df)} rows from: {filepath}')
    df_list.append(df)

# ignore_index=True creates clean, sequential row numbers
merged_df = pd.concat(df_list, ignore_index=True)
merged_df.head()

Loaded 10008 rows from: /Users/nhut/Library/CloudStorage/GoogleDrive-dhnhut@gmail.com/My Drive/0. 📌 UoA/0. Research Project/Datasets/CrisisLexT6/CrisisLexT6/2012_Sandy_Hurricane/2012_Sandy_Hurricane-ontopic_offtopic.csv
Loaded 10031 rows from: /Users/nhut/Library/CloudStorage/GoogleDrive-dhnhut@gmail.com/My Drive/0. 📌 UoA/0. Research Project/Datasets/CrisisLexT6/CrisisLexT6/2013_Alberta_Floods/2013_Alberta_Floods-ontopic_offtopic.csv
Loaded 10012 rows from: /Users/nhut/Library/CloudStorage/GoogleDrive-dhnhut@gmail.com/My Drive/0. 📌 UoA/0. Research Project/Datasets/CrisisLexT6/CrisisLexT6/2013_Boston_Bombings/2013_Boston_Bombings-ontopic_offtopic.csv
Loaded 9992 rows from: /Users/nhut/Library/CloudStorage/GoogleDrive-dhnhut@gmail.com/My Drive/0. 📌 UoA/0. Research Project/Datasets/CrisisLexT6/CrisisLexT6/2013_Oklahoma_Tornado/2013_Oklahoma_Tornado-ontopic_offtopic.csv
Loaded 10033 rows from: /Users/nhut/Library/CloudStorage/GoogleDrive-dhnhut@gmail.com/My Drive/0. 📌 UoA/0. Research Proje

,tweet,label,tweet_id,event_type,country,year,meta
0,I've got enough candles to supply a Mexican fa...,off-topic,262596552399396864,hurricane,US,2012,"{""file_name"": ""2012_Sandy_Hurricane/2012_Sandy..."
1,Sandy be soooo mad that she be shattering our ...,on-topic,263044104500420609,hurricane,US,2012,"{""file_name"": ""2012_Sandy_Hurricane/2012_Sandy..."
2,@ibexgirl thankfully Hurricane Waugh played it...,off-topic,263309629973491712,hurricane,US,2012,"{""file_name"": ""2012_Sandy_Hurricane/2012_Sandy..."
3,@taos you never got that magnificent case of B...,off-topic,263422851133079552,hurricane,US,2012,"{""file_name"": ""2012_Sandy_Hurricane/2012_Sandy..."
4,"I'm at Mad River Bar &amp; Grille (New York, N...",off-topic,262404311223504896,hurricane,US,2012,"{""file_name"": ""2012_Sandy_Hurricane/2012_Sandy..."


In [26]:
merged_df.to_csv( 'Cleaned_CrisisLexT6.csv', index=False)